# Fabric IQ — Generate Ontology Data
Loads the ontology package's instance data into the **lakehouse** (Delta tables) and its
time-series data into the **eventhouse** (Kusto tables), using the Fabric IQ accelerator library.
The `.whl` and `.iq` files were uploaded to `Files/` by the deployer.

In [ ]:
# Install the Fabric IQ Ontology Accelerator package (uploaded to the lakehouse Files/).
# Use a subprocess pip install instead of the `%pip` magic: `%pip` restarts the kernel
# and is unreliable in job-mode (deployer-triggered) runs — that surfaces as
# System_Cancelled_Session_Statements_Failed. subprocess installs in-place without
# tearing down the live Spark session.
import subprocess, sys
_whl = "/lakehouse/default/Files/{{WHL_FILENAME}}"
print(f"Installing {_whl} ...")
_p = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _whl],
                    capture_output=True, text=True)
print((_p.stdout or "")[-2000:])
if _p.returncode != 0:
    print((_p.stderr or "")[-3000:])
    raise RuntimeError(f"pip install failed (exit {_p.returncode}) — see stderr above")
print("Accelerator library installed.")


In [ ]:
from fabricontology.generate_data import generate_instance_data, generate_events_data
from notebookutils import mssparkutils

ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/{{IQ_FILENAME}}"
LAKEHOUSE_SCHEMA      = "{{LAKEHOUSE_SCHEMA}}"
EVENTHOUSE_CLUSTER_URI = "{{EVENTHOUSE_CLUSTER_URI}}"
EVENTHOUSE_DATABASE    = "{{EVENTHOUSE_DATABASE}}"

print(f"Package : {ONTOLOGY_PACKAGE_PATH}")
print(f"Lakehouse schema : {LAKEHOUSE_SCHEMA}")
print(f"Eventhouse : {EVENTHOUSE_CLUSTER_URI} / {EVENTHOUSE_DATABASE}")

In [ ]:
# Shift the packaged time-series so the newest event lands ~yesterday.
# The sample events in the .iq were generated at package-build time; without a
# shift, agent questions about recent periods find no rows in range. A single
# global day-offset is applied to every events_data member, preserving relative
# order and time-of-day. Idempotent: re-running shifts by ~0 days.
import csv as _csv
import io as _iocsv
import os as _os
import zipfile as _zipf
from datetime import datetime as _dt, timedelta as _td

_TS_FORMATS = ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d")

def _parse_ts(_v):
    for _f in _TS_FORMATS:
        try:
            return _dt.strptime(_v, _f), _f
        except ValueError:
            continue
    return None, None

_zin = _zipf.ZipFile(ONTOLOGY_PACKAGE_PATH)
_members = {n: _zin.read(n) for n in _zin.namelist()}
_zin.close()

_gmax = None
for _n, _data in _members.items():
    if not (_n.startswith("events_data/") and _n.lower().endswith(".csv")):
        continue
    _rd = _csv.DictReader(_iocsv.StringIO(_data.decode("utf-8")))
    for _row in _rd:
        for _col, _val in _row.items():
            if _col and "timestamp" in _col.lower() and _val:
                _t, _ = _parse_ts(_val)
                if _t and (_gmax is None or _t > _gmax):
                    _gmax = _t

if _gmax is None:
    print("No timestamp columns found in events_data - skipping shift.")
else:
    _delta = _td(days=(_dt.utcnow() - _td(days=1) - _gmax).days)
    if _delta.days <= 0:
        print(f"Events already current (max {_gmax:%Y-%m-%d}) - no shift needed.")
    else:
        for _n in list(_members):
            if not (_n.startswith("events_data/") and _n.lower().endswith(".csv")):
                continue
            _rd = _csv.DictReader(_iocsv.StringIO(_members[_n].decode("utf-8")))
            _out = _iocsv.StringIO()
            _wr = _csv.DictWriter(_out, fieldnames=_rd.fieldnames, lineterminator="\n")
            _wr.writeheader()
            for _row in _rd:
                for _col in _rd.fieldnames:
                    if _col and "timestamp" in _col.lower() and _row[_col]:
                        _t, _f = _parse_ts(_row[_col])
                        if _t:
                            _row[_col] = (_t + _delta).strftime(_f)
                _wr.writerow(_row)
            _members[_n] = _out.getvalue().encode("utf-8")
        _tmp = ONTOLOGY_PACKAGE_PATH + ".shifted"
        with _zipf.ZipFile(_tmp, "w", _zipf.ZIP_DEFLATED) as _zout:
            for _n, _data in _members.items():
                _zout.writestr(_n, _data)
        _os.replace(_tmp, ONTOLOGY_PACKAGE_PATH)
        print(f"Shifted events_data timestamps forward by {_delta.days} days (was max {_gmax:%Y-%m-%d}).")

In [ ]:
# Create the instance (static) Delta tables in the default lakehouse.
instance_result = generate_instance_data(
    spark,
    ontology_package_path=ONTOLOGY_PACKAGE_PATH,
    database=LAKEHOUSE_SCHEMA,
    mode="overwrite",
)
print("Lakehouse tables created:")
for k, v in (instance_result or {}).items():
    print(f"  {k} -> {v}")

In [ ]:
# Create the time-series Kusto tables in the eventhouse (best-effort).
# A freshly-created Eventhouse can take a short while before its default KQL
# database accepts writes; retry a few times. A persistent Kusto hiccup must NOT
# cancel the whole session — the lakehouse tables + ontology can still proceed,
# so we capture the outcome instead of raising.
import json as _json, time as _time
from notebookutils import mssparkutils

events_result = {}
events_error = ""
for _attempt in range(4):
    try:
        access_token = mssparkutils.credentials.getToken(EVENTHOUSE_CLUSTER_URI)
        events_result = generate_events_data(
            spark,
            ontology_package_path=ONTOLOGY_PACKAGE_PATH,
            eventhouse_cluster_uri=EVENTHOUSE_CLUSTER_URI,
            eventhouse_database=EVENTHOUSE_DATABASE,
            access_token=access_token,
        )
        events_error = ""
        break
    except Exception as _e:  # noqa: BLE001
        events_error = f"{type(_e).__name__}: {_e}"
        print(f"Eventhouse write attempt {_attempt + 1}/4 failed: {events_error[:300]}")
        _time.sleep(20)

print("Eventhouse tables created:" if events_result else "Eventhouse tables NOT created.")
for k, v in (events_result or {}).items():
    print(f"  {k} -> {v}")

# Record the outcome so the deployer can read it back.
_summary = {
    "lakehouseTables": list((instance_result or {}).values()),
    "eventhouseTables": list((events_result or {}).values()),
    "eventhouseError": events_error,
}
try:
    with open("/lakehouse/default/Files/load_data_result.json", "w") as _f:
        _json.dump(_summary, _f)
except Exception as _fe:  # noqa: BLE001
    print(f"Could not write load_data_result.json: {_fe}")

print("\nData load complete (lakehouse always; eventhouse best-effort).")
print(_json.dumps(_summary, indent=2))
